# 01 — Merge and clean

**Takes in:** raw outcome and covariate CSVs from `00_pull.ipynb`.

**Does:** validated one-to-one tract merge, row-loss diagnostics, analysis variables, and missingness audit.

**Outputs:** `data/analysis_sample.csv`, `output/table2_missing_cells.tex`

In [1]:
import importlib
import os
import sys
import pandas as pd

sys.path.insert(0, os.path.abspath("."))
import utils
importlib.reload(utils)

pd.set_option("display.max_columns", 40)

## Functions

In [2]:
def read_outcomes(data_dir):
    """Read the pooled p25 outcome, its uncertainty, count, and geography."""
    keep = utils.ID_COLS + ["cz", "czname", utils.MOBILITY_VAR, utils.COUNT_VAR]
    return pd.read_csv(os.path.join(data_dir, "tract_outcomes_simple.csv"), usecols=keep)


def read_covariates(data_dir):
    """Read covariates, dropping duplicate commuting-zone columns."""
    cov = pd.read_csv(os.path.join(data_dir, "tract_covariates.csv"))
    return cov.drop(columns=["cz", "czname"])


def merge_with_diagnostics(outcomes, covariates):
    print("rows before merge -- outcomes: {:,} | covariates: {:,}".format(len(outcomes), len(covariates)))
    merged = outcomes.merge(covariates, on=utils.ID_COLS, how="inner", validate="one_to_one")
    print("rows after inner join: {:,}".format(len(merged)))
    print("outcomes rows dropped: {:,} | covariate rows dropped: {:,}".format(
        len(outcomes) - len(merged), len(covariates) - len(merged)))
    return merged


def drop_unusable(df):
    out = df.dropna(subset=[utils.MOBILITY_VAR, "popdensity2000"]).copy()
    print("dropped {:,} tracts missing mobility or density; {:,} remain".format(len(df) - len(out), len(out)))
    return out


def add_analysis_vars(df):
    out = df.copy()
    out["density_cat"] = pd.cut(out["popdensity2000"], bins=utils.DENSITY_BINS,
                                   labels=utils.DENSITY_LABELS, right=False)
    national_median = out[utils.MOBILITY_VAR].median()
    out["above_median_mobility"] = out[utils.MOBILITY_VAR] > national_median
    print("national tract median: {:.3f}".format(national_median))
    return out


def missing_cells_table(df, fields):
    rural_mask = df["popdensity2000"] < utils.RURAL_CUTOFF
    rows = []
    for col, label in fields.items():
        rows.append({"label": label, "n_missing": int(df[col].isna().sum()),
                     "pct_missing": 100 * df[col].isna().mean(),
                     "pct_missing_rural": 100 * df.loc[rural_mask, col].isna().mean()})
    return pd.DataFrame(rows).sort_values("pct_missing", ascending=False)


def write_missing_latex(table, out_path):
    lines = ["\\begin{tabular}{lrrr}", "\\toprule",
             "Field & N missing & \\% missing & \\% missing (rural) \\\\", "\\midrule"]
    for _, row in table.iterrows():
        lines.append("{} & {:,} & {:.1f} & {:.1f} \\\\".format(
            row["label"], row["n_missing"], row["pct_missing"], row["pct_missing_rural"]))
    lines += ["\\bottomrule", "\\end{tabular}"]
    with open(out_path, "w") as handle:
        handle.write("\n".join(lines) + "\n")
    print("wrote", out_path)

## Read, merge, and derive analysis variables

In [3]:
outcomes = read_outcomes(utils.DATA_DIR)
covariates = read_covariates(utils.DATA_DIR)
print("outcomes:  {:,} rows x {} cols".format(*outcomes.shape))
print("covariates: {:,} rows x {} cols".format(*covariates.shape))

merged_raw = merge_with_diagnostics(outcomes, covariates)
analysis = add_analysis_vars(drop_unusable(merged_raw))
print("\ntracts per density category:")
print(analysis["density_cat"].value_counts().sort_index().to_string())

rural = analysis[analysis["popdensity2000"] < utils.RURAL_CUTOFF]
print("\nrural tracts: {:,}".format(len(rural)))
print("with at least {:,} children: {:,} ({:.1%})".format(
    utils.MIN_CHILD_COUNT, int((rural[utils.COUNT_VAR] >= utils.MIN_CHILD_COUNT).sum()),
    (rural[utils.COUNT_VAR] >= utils.MIN_CHILD_COUNT).mean()))

outcomes:  73,278 rows x 7 cols
covariates: 74,044 rows x 36 cols
rows before merge -- outcomes: 73,278 | covariates: 74,044
rows after inner join: 73,199
outcomes rows dropped: 79 | covariate rows dropped: 845
dropped 1,241 tracts missing mobility or density; 71,958 remain
national tract median: 0.425

tracts per density category:
density_cat
Rural\n(<100)            17978
Small town\n(100-500)    12777
Suburban\n(500-2k)       23101
Urban\n(2k-10k)          15520
Dense urban\n(10k+)       2582

rural tracts: 17,978
with at least 200 children: 15,186 (84.5%)


## Missingness audit

Reported on the matched sample before rows are dropped. Rural percentages condition on an observed density below 100, so rural density missingness is necessarily zero.

In [4]:
key_fields = {utils.MOBILITY_VAR: "Upward mobility (income rank)",
              utils.COUNT_VAR: "Children behind mobility estimate",
              "popdensity2000": "Population density, 2000"}
key_fields.update(utils.COVARIATES)
miss = missing_cells_table(merged_raw, key_fields)
miss[["label", "n_missing", "pct_missing", "pct_missing_rural"]]

,label,n_missing,pct_missing,pct_missing_rural
6,"Annual job growth, 2004-13",2531,3.457698,1.055972
1,Children behind mobility estimate,1249,1.706307,1.674235
0,Upward mobility (income rank),1189,1.624339,1.635936
8,Mean 3rd-grade math score,1109,1.515048,1.362368
3,Share single-parent households,910,1.243186,0.716748
9,Mean household income,893,1.219962,0.749576
5,Mean commute time,882,1.204934,0.760519
4,Share below poverty line,880,1.202202,0.722219
11,Share with a BA or higher,852,1.163950,0.683920
10,Share of adults employed,851,1.162584,0.683920


Missingness is not uniform: job growth has the highest overall missing rate, while Census mail-return missingness is higher among rural tracts. Models use complete cases and report the resulting row loss.

## Write outputs

In [5]:
os.makedirs(utils.OUT_DIR, exist_ok=True)
write_missing_latex(miss, utils.output_path("table2_missing_cells.tex"))
out_path = utils.data_path("analysis_sample.csv")
analysis.to_csv(out_path, index=False)
print("wrote {:,} rows x {} cols to {}".format(len(analysis), analysis.shape[1], out_path))

wrote /Users/maxfortner/Documents/Dartmouth/QSS20/qss20-rural-mobility/output/table2_missing_cells.tex


wrote 71,958 rows x 42 cols to /Users/maxfortner/Documents/Dartmouth/QSS20/qss20-rural-mobility/data/analysis_sample.csv
